## Bronze - VRA (Voo Regular Ativo)

Lê 31 CSVs mensais de volume /Volumes/voebem/bronze/arquivos/vra/ e materializa voebem.bronze.vra  

Regras da camada Bronze:  

*   **nada de tipagem** - tudo string, exatamente como veio o arquivo;  
*   **nada de filtro** - nenhuma linha é descartada;  
*   **colunas de auditoria** - de qual arquivo e quando foi ingerido;  
* **idempotente** - rodar duas vezes não duplica.  

In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra"

## Leitura

Quatro opções carregam quatro problemas do arquivo:



| opção | resolve |
|---|---|
| `sep=";"` | separador brasileiro, não vírgula |
| `skipRows=1` | a 1ª linha é `Atualizado em: <data>`, não o cabeçalho — e o `B0M EF BB BF` mora nela, some junto |
| `header=true` | a 2ª linha (a primeira que sobra) é o cabeçalho de verdade |
| `inferSchema` **desligado** (default) | bronze não tipa: tudo chega como `string` |


In [0]:
bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", 1)          # Descarta a primeira linha
    .option("quote", '"')           # Define o caractere de
    .option("escape", '"')           # escape como '"'
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE") # bronze não descata linha nenhuma
    .load(CAMINHO)    
)

print("colunas lidas do arquivo:")
for c in bruto.columns:
    print(f"  {c!r}")


# Nomes de coluna: o Delta não aceita espaço

`ICAO Empresa Aérea` é um nome de coluna válido em CSV e **inválido** em Delta — espaço está na lista de caracteres proibidos (` ,;{}()\n\t= `).

Então normalizamos o **nome**. Repare que isso não fere a regra da bronze: o que a bronze preserva é o **valor** e a **granularidade**, não a grafia do cabeçalho. Nenhuma coluna é somada, removida, filtrada ou convertida.

O mapa fica explícito no código — nada de `regexp_replace` mágico, para que a correspondência com o arquivo original seja auditável.

In [0]:
RENOMEAR = {
    "ICAO Empresa Aérea": "icao_empresa",
    "Número Voo": "numero_voo",
    "Código Autorização (DI)": "codigo_di",
    "Código Tipo Linha": "codigo_tipo_linha",
    "ICAO Aeródromo Origem": "icao_origem",
    "ICAO Aeródromo Destino": "icao_destino",
    "Partida Prevista": "partida_prevista",
    "Partida Real": "partida_real",
    "Chegada Prevista": "chegada_prevista",
    "Chegada Real": "chegada_real",
    "Situação Voo": "situacao_voo",
    "Código Justificativa": "codigo_justificativa"
}

faltando = [c for c in RENOMEAR if c not in bruto.columns]
assert not faltando, f"Coluna esperada não encontrada no CSV: {faltando}"

renomeado = bruto.select(
    *[F.col(f"{origem}").cast("string").alias(novo) for origem, novo in RENOMEAR.items()]
)



In [0]:
display(renomeado)

# Auditoria

Duas colunas que o arquivo não tem e a tabela precisa ter:
`_arquivo_origem` (de qual CSV a linha veio — `_metadata` é uma coluna oculta que o Spark expõe em qualquer leitura de arquivo) e `_ingerido_em`.

In [0]:
bronze = renomeado.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")
).withColumn(
    "_ingerido_em", F.current_timestamp()
)

# Escrita idempotente

Estratégia: **full refresh determinístico** — `mode("overwrite")` sobre o conjunto inteiro de arquivos.

Por que essa e não um `append` com deduplicação:

1. A fonte é **imutável e completa**: o volume tem os 31 arquivos do mês fechado, e a ANAC republica o mês inteiro quando corrige algo[cite: 5]. A entrada define o estado final — logo o destino pode ser derivado inteiro dela[cite: 5].
2. `append` exigiria uma chave de negócio para deduplicar[cite: 5, 6]. O VRA **não tem chave natural única** (o mesmo voo pode repetir legitimamente na mesma data — veja o código DI "Etapa de Voo Duplicada")[cite: 5, 6]. Deduplicar no bronze seria decidir regra de negócio na camada errada[cite: 6].
3. `overwrite` no Delta é **atômico**: ou a versão nova aparece inteira, ou a antiga continua valendo[cite: 6]. Ninguém lê tabela pela metade[cite: 6].
4. O histórico não se perde: cada `overwrite` gera uma versão nova no log do Delta, e a anterior continua acessível por time travel (marco-04)[cite: 6, 7].

O que muda entre duas execuções: só `_ingerido_em`[cite: 7]. O **conjunto de linhas** é idêntico — é isso que a validação prova[cite: 7].

In [0]:
(
    bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA)
)

print(f"{TABELA}: {spark.table(TABELA).count():,} linhas")

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {TABELA} IS
    'Bronze - VRA (Voo Regular Ativo) da ANAC, 31 meses (jan/2024 a jul/2026).
    Dado bruto: todas as colunas string, nenhuma linha descartada.
    Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/vra/.'
""")

In [0]:
display(
    spark.sql(f"""
    SELECT
        _arquivo_origem,
        COUNT(*) AS linhas,
        MAX (_ingerido_em) AS data_ingerido
    FROM {TABELA}
    GROUP BY _arquivo_origem
    ORDER BY _arquivo_origem
    """)
)